# قالب نوت‌بوک GPU — فاز ۷

> بند 7.8.2 سند `doc/WBS-phase7-modeling.md`. **کپی کنید** به `notebooks/gpu/GPUxx_{model}_{level}.ipynb` و پر کنید — این فایل خودش هرگز مستقیم اجرا/ویرایش نمی‌شود.
>
> **قاعده‌ی حاکم (بند 7.8.4):** کد اصلی در نوت‌بوک نوشته نمی‌شود؛ نوت‌بوک فقط `src/` را import می‌کند (مخزن با `git clone` داخل کولب کشیده می‌شود). قاعده‌ی «`notebooks/` = روایت، `src/` = حقیقت» (`AGENTS.md`) در کولب هم برقرار است.
>
> این ۸ سلول **اجباری‌اند** و ترتیبشان عوض نمی‌شود. جاهای `<TODO: ...>` را پیش از اجرا پر کنید.

## سلول ۱ — نصب نسخه‌های پین‌شده
نتیجه‌ی کولب باید با اجرای محلی قابل‌قیاس باشد؛ نسخه‌های پین‌نشده این را نقض می‌کنند.

In [ ]:
# <TODO: آدرس remote مخزن را جایگزین کنید>
GIT_REMOTE_URL = "<TODO: git@github.com:ORG/REPO.git یا مشابه>"

!git clone --depth 1 "$GIT_REMOTE_URL" repo
%cd repo

# requirements-gpu.lock هنگام اولین کار GPU (خانواده‌ی ۷، بند 7.8) ساخته و پین می‌شود.
!pip install -q -r requirements-gpu.lock

## سلول ۲ — بارگذاری داده‌ی پردازش‌شده + `cv_folds.json`
از Google Drive یا Kaggle Dataset — هر دو باید همان فایل‌های `data/processed/` مخزن محلی باشند.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# <TODO: مسیر واقعی پروژه روی Drive>
DRIVE_DATA_DIR = "/content/drive/MyDrive/<TODO: PATH_TO_PROJECT>/data/processed"

import pathlib
import shutil

pathlib.Path("data/processed").mkdir(parents=True, exist_ok=True)

# <TODO: فایل‌های سطح داده‌ی این مدل را اضافه/جایگزین کنید (مثلاً person_features_v1.parquet برای L5)>
FILES_NEEDED = ["features_A_v1.parquet", "cv_folds.json"]
for name in FILES_NEEDED:
    shutil.copy(f"{DRIVE_DATA_DIR}/{name}", f"data/processed/{name}")

## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)
**اگر هش‌ها نخوانند، نوت‌بوک باید همین‌جا خطا بدهد و متوقف شود.** بدون این assert، هیچ ادعایی درباره‌ی «همان fold و همان داده» قابل‌اثبات نیست.

In [ ]:
from pathlib import Path

from src.cv import load_cv_folds, sha256_file

# مقادیر زیر را از doc/data_manifest.md کپی کنید — دستی، نه از فایل محلی، تا از خودِ سند سرچشمه بگیرند.
EXPECTED_CV_FOLDS_HASH = "<TODO: از بخش «قفل فاز ۷» doc/data_manifest.md>"
EXPECTED_DATA_SNAPSHOT_HASH = "<TODO: هش snapshot دادهٔ سطح این مدل از doc/data_manifest.md>"
DATA_SNAPSHOT_PATH = Path("data/processed/features_A_v1.parquet")  # <TODO: مسیر واقعی سطح داده

_, cv_folds_hash = load_cv_folds()
assert cv_folds_hash == EXPECTED_CV_FOLDS_HASH, (
    f"cv_folds_hash نامنطبق: {cv_folds_hash} != {EXPECTED_CV_FOLDS_HASH} — "
    "این run از جدول مقایسه‌ی فاز ۷ حذف خواهد شد، ادامه ندهید."
)

data_snapshot_hash = sha256_file(DATA_SNAPSHOT_PATH)
assert data_snapshot_hash == EXPECTED_DATA_SNAPSHOT_HASH, (
    f"data_snapshot_hash نامنطبق: {data_snapshot_hash} != {EXPECTED_DATA_SNAPSHOT_HASH}"
)

print("✅ cv_folds_hash و data_snapshot_hash هر دو تأیید شدند")

## سلول ۴ — بذر تصادفی سراسری + قطعیت PyTorch

In [ ]:
from src.config import set_global_seed

set_global_seed()

import torch

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# ⚠️ روی GPU قطعیت کامل تضمین‌شدنی نیست — به همین دلیل قاعده‌ی سه seed (A7) برای مدل‌های عصبی الزامی است، نه اختیاری (بند 7.7.4).

## سلول ۵ — سخت‌افزار
زمان‌های اجرا فقط با دانستن سخت‌افزار قابل تفسیرند.

In [ ]:
!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available(), "| CUDA version:", torch.version.cuda)

## سلول ۶ — ردیابی MLflow جدا برای اجرای GPU
بعداً در مرحله‌ی بازگشت (سلول ۸ + بند 7.8.3) با `mlruns/` محلی ادغام می‌شود.

In [ ]:
import mlflow

mlflow.set_tracking_uri("./mlruns_gpu")
COMPUTE = "colab"  # یا "kaggle" — بند 7.7.2، tag اجباری

## سلول ۷ — بدنه‌ی آزمایش (تنظیم + برازش + ارزیابی)
فقط `src/` را import کنید — منطق مدل اینجا نوشته نمی‌شود (بند 7.8.4).

In [ ]:
from src.models.tracking import start_model_run

# <TODO: مقادیر واقعی این اجرا — از src/models/registry.py و برنامه‌ی family 7 بند 7.16>
with start_model_run(
    family="F07",
    model="<TODO: model_id>",
    level="<TODO: L1|L4|L5>",
    target="rho",
    feature_set="<TODO: FSxxx>",
    tau=0.10,
    stage="S2",
    seed=42,
    data_snapshot_hash=data_snapshot_hash,
    cv_folds_hash=cv_folds_hash,
    compute=COMPUTE,
) as run:
    run_name = run.info.run_name
    # <TODO: فراخوانی src/models/<family>/<model>.py — تنظیم Optuna + برازش نهایی + ثبت metricها>
    pass

## سلول ۸ — بسته‌بندی خروجی برای بازگشت به مخزن محلی
طبق بند 7.8.3 چرخه‌ی رفت‌وبرگشت: این zip + خودِ نوت‌بوک اجراشده (با خروجی‌ها، حتی اگر شکست خورده باشد) باید به مخزن برگردند.

In [ ]:
import json
import pathlib

!zip -rq mlruns_gpu.zip mlruns_gpu

pathlib.Path("reports/gpu").mkdir(parents=True, exist_ok=True)
summary = {"run_name": run_name, "compute": COMPUTE, "cv_folds_hash": cv_folds_hash,
          "data_snapshot_hash": data_snapshot_hash}
with open(f"reports/gpu/{run_name}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

from google.colab import files

files.download("mlruns_gpu.zip")
files.download(f"reports/gpu/{run_name}.json")

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3، خارج از این نوت‌بوک)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb   (حتی اگر آزمایش شکست خورده — شکست هم داده است)
۲. mlruns_gpu.zip →  از حالت فشرده خارج و در mlruns/ محلی ادغام شود
۳. reports/gpu/{run_name}.json → بایگانی می‌شود
۴. کارت مدل (src/models/cards.py) در reports/models/{model_id}.md نوشته/تکمیل شود
۵. mlflow ui محلی باز و run جدید با tag compute=colab/kaggle تأیید شود
۶. اگر نیمه‌کاره ماند: checkpoint هر ۱۰ trial در optuna_studies/*.db روی Drive، ادامه‌پذیر
```